# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook demonstrates loading, exploring, and processing the FAIR² dataset using the [`mlcroissant`](https://pypi.org/project/mlcroissant/) library, showcasing best practices for working with Croissant schemas and datasets.

### Dataset Source
This dataset is described by a Croissant schema at:
https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json

In [ ]:
# Install the required mlcroissant library
!pip install mlcroissant

## 1. Data Loading
Load metadata and explore available record sets and fields using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
import warnings
warnings.filterwarnings('ignore')

# Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)
# The .metadata object contains dataset metadata; access fields as attributes
meta = dataset.metadata
print(f"Dataset: {meta.name}\n\n{meta.description}")

## 2. Data Overview
Let's review all available record sets with their record set `@id`. For each record set, we list the available fields and their respective `@id` and names.

In [ ]:
# List all record sets and their fields using @id references
record_sets = list(dataset.metadata.record_sets)
if not record_sets:
    print('No record sets found in the dataset metadata.')
else:
    for rs in record_sets:
        print(f"\nRecordSet Name: {rs.name}\n@id: {rs.id}")
        print("Fields:")
        for field in rs.fields:
            print(f"    - {field.name}: @id = {field.id}")

## 3. Data Extraction
We extract all data from each record set into separate pandas DataFrames for easy analysis. All record set and field references use their `@id` as required.

In [ ]:
# Extract all dataframes by RecordSet @id
dataframes = {}
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded records for RecordSet @id: {record_set_id} (shape: {df.shape})")
    else:
        print(f"No records found for RecordSet @id: {record_set_id}")

# Print columns of the first record set as an example
if len(dataframes) > 0:
    first_rs = list(dataframes.keys())[0]
    print(f"\nColumns in RecordSet @id '{first_rs}':")
    print(dataframes[first_rs].columns.tolist())
    display(dataframes[first_rs].head())

## 4. Exploratory Data Analysis (EDA)
We'll now perform some basic EDA. This includes filtering numeric records, normalization, and grouping for one record set. All operations reference columns and record sets by their `@id`.

#### **Please replace the `record_set_id`, `numeric_field_id`, and `group_field_id` below according to actual dataset content from the overview.**

In [ ]:
# --- Update these IDs for your dataset after running above cells! ---
# Use your actual record set, numeric field, and group field @id (see previous overview output):
record_set_id = list(dataframes.keys())[0] if dataframes else None
# Example: numeric_field_id = 'https://api.app.sen.science/frontiers/7862866/fields/age'
# Example: group_field_id = 'https://api.app.sen.science/frontiers/7862866/fields/sex'
numeric_field_id = None
group_field_id = None

# Try to guess a numeric field id
if record_set_id:
    df = dataframes[record_set_id]
    possible_numeric = [col for col in df.columns if df[col].dtype in ['int64', 'float64'] or (df[col].dtype == 'object' and pd.to_numeric(df[col], errors='coerce').notnull().all())]
    if possible_numeric:
        numeric_field_id = possible_numeric[0]

    possible_group = [col for col in df.columns if col != numeric_field_id and df[col].nunique() < 10]
    if possible_group:
        group_field_id = possible_group[0]

    if numeric_field_id is None:
        print('No numeric field detected for EDA. Please set numeric_field_id manually.')
    else:
        print(f'Numeric field selected: {numeric_field_id}')

    if group_field_id is None:
        print('No group field detected. Please set group_field_id manually if grouping is needed.')
    else:
        print(f'Group field selected: {group_field_id}')

if record_set_id and numeric_field_id:
    # Try converting to numeric, just in case
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    threshold = df[numeric_field_id].mean() if pd.notnull(df[numeric_field_id].mean()) else 0
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records where '{numeric_field_id}' > {threshold} (mean):")
    display(filtered_df.head())

    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized '{numeric_field_id}' for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped mean of '{numeric_field_id}' by '{group_field_id}':")
        display(grouped_df.head())

## 5. Visualization
Let's visualize the distribution of the selected numeric field and relationships by group, if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_set_id and numeric_field_id and numeric_field_id in dataframes[record_set_id].columns:
    df = dataframes[record_set_id]
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id], kde=True)
    plt.xlabel(numeric_field_id)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.show()

    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(8,4))
        sns.boxplot(data=df, x=group_field_id, y=numeric_field_id)
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.show()


## 6. Conclusion
In this notebook, we demonstrated how to load and process a Croissant-described dataset using `mlcroissant`. All data operations referenced entities by their `@id` and followed robust data science workflows. Explore further by updating the field and record set IDs to match the topics of your analysis.